## vLLM部署方案流程

In [ ]:
安装vllm/cuda(GPU) /工具包/等环境（window需要通过WSL2+Ubuntu进行部署）

https://docs.vllm.ai/en/stable/getting_started/installation/

In [ ]:
pip install vllm

pip install -U huggingface_hub #vllm 自带，这里更新为最新版本，可以不需要

## 准备模型

注意支持的模型列表

多模态/文本模型说明及模型列表：https://docs.vllm.ai/en/stable/models/supported_models/

接下来，以Qwen3-8B模型为案例

1、下载文件

In [ ]:
export HF_ENDPOINT = https://hf-mirror.com 

In [ ]:
huggingface-cli download --resume-download Qwen/Qwen3-8B --local-dir{local_path} --local-dir-use-symlinks False

Qwen/Qwen3-8B 为huggingface上的模型名字

local-dir{local_path}中的{local_path}可替换为本地路径

local-dir-use-symlinks False：表示参数禁用文件软链接，下载路径所见及所得

### modelscope 下载方式(国内源，会比较快)

In [ ]:

pip install modelscope
modelscope download --model Qwen/Qwen3-8B --local_dir{local_path}

## 部署OpenAI API服务

In [ ]:
CUDA_VISIBLE_DEVICRS=0,1 vllm serve {local_path} \
    --api-key abc123  \
    --served-model-name Qwen/Qwen3-8B \
    --max_model_len 4096 \
    --tensor-parallel-size 2 \
    --port 7890

最大处理长度（上下文+输出）4096  

GPU卡数2 ,

指定端口7890  

local需要改为本地路径 当写成一行且只有一个GPU的时候，可以不需要\ 和CUDA_VISIBLE_DEVICRS=

当只有一张卡的时候，不要填写并行数量和0，1


通过 --chat-template 可以指定对话模板，主流的大模型可以不用管该模板。这是在微调训练时用的模板，对于llm唯一

## 服务验证

In [ ]:
curl http://{ip}:7890/v1/completions -H "Authorization: Bearer abc123"

In [ ]:
import requests

url = f"http://{ip}:7890/v1/completions"
headers = {"Authorization": "Bearer abc123"}
response = requests.get{url,headers=headers}
print(response.json())

## 调用对话服务

In [ ]:
from openai import OpenAI
client = OpenAI(
    base_url= f"http:{ip}:7890/v1",
    api_key= "abc123",
)

completion = client.chat.completions.create(
    model="Qwen3-8B"
    messages=[
        {"role": "user","content":"hellow"}
    ]
)

print(compile.choices[0].message)

部署在本地的化{ip}填写本地地址

## Autodl部署并暴露

在容器实例中选择自定义服务，便可在服务器外调用

In [ ]:
curl http://localhost:7890/v1/models

curl http://localhost:7890/v1/chat/completions \
    -H "Content - Type: application/json"\
    -d '{
        "model":"Qwen3-8B",
        "messages":[
        {"role": "user","content":"hellow"}
        ]
    }'
